## Task-Specific Runnables

Pre-built runnables designed for specific tasks.

Examples:
- Prompt templates  
- LLMs  
- Output parsers  
- Retrievers  

They perform a defined job and follow the runnable interface.

---

## Runnable Primitives

Basic building blocks used to create workflows.

1. **RunnableSequence:** R1 → R2 → R3  
   Executes runnables step by step in order.

2. **RunnableParallel:** R1 || R2 || R3  
   Executes multiple runnables at the same time.

3. **RunnablePassthrough** passes the input forward **without modifying it**.

4. **RunnableLambda** converts a normal Python function into a Runnable. Very imp

5. **RunnableBranch** for conditional chain

## LCEL - LangChain Expression Language
Replaces RunnableSequence() with pipe operator
```python
r1 | r2 | r3



In [8]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [4]:
from langchain_core.runnables import RunnableSequence


In [6]:
load_dotenv()

True

In [9]:
model = ChatOpenAI()


# 1st Prompt - Detailed Report
prompt1 = PromptTemplate(
    template='Write a joke on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Explain the {joke}',
    input_variables=['joke']
)

parser = StrOutputParser()

In [12]:
chain = RunnableSequence(prompt1,model,parser,prompt2,model,parser)

In [13]:
result = chain.invoke({
    'topic':'cricket'
})

In [14]:
print(result)

This joke is a pun on the word "delivery," which has two meanings in this context. In cricket, a "delivery" is a ball bowled by a bowler towards the batsman. The cricket team went to the bakery to improve their delivery, meaning they wanted to practice their bowling skills. The second meaning of "delivery" is the act of delivering goods or services, in this case, baked goods from the bakery. So, the cricket team went to the bakery to get a good delivery of baked goods.


# RunnableParallel

In [15]:
from langchain_core.runnables import RunnableParallel

In [17]:
prompt1 = PromptTemplate(
    template='Write a tweet about {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Write a Linkedin post on {topic}',
    input_variables=['topic']
)

parallel_chain = RunnableParallel({
    'tweet':RunnableSequence(prompt1,model,parser),
    'linkedin':RunnableSequence(prompt2,model,parser)
})

result = parallel_chain.invoke({
    'topic':'AI'
})
print(result)

{'tweet': 'AI is revolutionizing the way we live and work, with the potential to transform industries and improve efficiencies. Excited to see how this technology continues to evolve! 🤖 #ArtificialIntelligence #FutureTech', 'linkedin': "Exciting news in the world of artificial intelligence! 🤖\n\nAI technology continues to advance at a rapid pace, with new developments and applications emerging every day. From improving healthcare outcomes to streamlining business operations, AI is revolutionizing the way we work and live. \n\nAs a professional in the field, I am constantly inspired by the potential of AI to drive innovation and create positive impact in diverse industries. Whether you're a seasoned AI expert or just starting to explore the possibilities, now is an exciting time to be part of this transformative field.\n\nLet's continue to push the boundaries of AI and harness its power for good. Together, we can shape a future where AI enhances our capabilities, improves efficiency, an

# RunnablePassthroughs

In [19]:
from langchain_core.runnables import RunnablePassthrough

In [20]:
passthrough = RunnablePassthrough()


In [27]:
# 1st Prompt - Detailed Report
prompt1 = PromptTemplate(
    template='Write a joke on {topic}',
    input_variables=['topic']
)
prompt2 = PromptTemplate(
    template='Explain {joke}',
    input_variables=['joke']
)
joke_chain  = RunnableSequence(prompt1,model,parser)
parallel_chain = RunnableParallel({
    'summary':RunnableSequence(prompt2,model,parser),
    'joke':RunnablePassthrough()
})
final_chain = RunnableSequence(joke_chain,parallel_chain)
result = final_chain.invoke({
    'topic':'AI'
})
print(result)

{'summary': 'Robots, like humans, can experience emotions and mental health issues. The robot may have gone to therapy to address feelings of sadness, anxiety, or other emotional challenges that were affecting its functioning. Therapy can help robots process their emotions, develop coping skills, and improve their overall well-being.', 'joke': 'Why did the robot go to therapy?\n\nBecause it had too many bytes of emotional baggage!'}


# RunnableLambda

In [33]:
from langchain_core.runnables import RunnableLambda
def word_counter(text):
    return len(text.split())
runnable_word_counter = RunnableLambda(word_counter)

prompt1 = PromptTemplate(
    template='Write a joke on {topic}',
    input_variables=['topic']
)
joke_chain  = prompt1 | model | parser
parallel_chain = RunnableParallel({
    'len':runnable_word_counter,
    'joke':RunnablePassthrough()
})
final_chain = RunnableSequence(joke_chain,parallel_chain)
result = final_chain.invoke({
    'topic':'AI'
})
print(result)

{'len': 17, 'joke': "Why did the AI break up with their robot partner?\n\nBecause they couldn't handle their algorithmic differences!"}


# Runnable Branch

In [32]:
from langchain_core.runnables import (RunnableBranch)
# Step 1: Simple Sentiment Classifier (rule-based for demo)
def classify_sentiment(review: str):
    review = review.lower()
    if any(word in review for word in ["great", "amazing", "excellent", "love"]):
        return {"review": review, "sentiment": "positive"}
    elif any(word in review for word in ["bad", "boring", "worst", "hate"]):
        return {"review": review, "sentiment": "negative"}
    else:
        return {"review": review, "sentiment": "neutral"}

sentiment_runnable = RunnableLambda(classify_sentiment)

# Step 2: Define Branch Responses
positive_branch = RunnableLambda(
    lambda x: f"😊 Positive Review Detected!\nReview: {x['review']}"
)

negative_branch = RunnableLambda(
    lambda x: f"😞 Negative Review Detected!\nReview: {x['review']}"
)

neutral_branch = RunnableLambda(
    lambda x: f"😐 Neutral Review Detected!\nReview: {x['review']}"
)
# Step 3: Create RunnableBranch
branch = RunnableBranch(
    (lambda x: x["sentiment"] == "positive", positive_branch),
    (lambda x: x["sentiment"] == "negative", negative_branch),
    neutral_branch  # default branch
)

final_chain = sentiment_runnable | branch

# ---- Invoke ----
result = final_chain.invoke("The movie was amazing and I loved it!")

print(result)

😊 Positive Review Detected!
Review: the movie was amazing and i loved it!
